In [ ]:
import matplotlib.pyplot as plt
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.indicators.igd_plus import IGDPlus
from pymoo.indicators.igd import IGD
from pymoo.indicators.gd import GD
import numpy as np
import pandas as pd

from matplotlib.animation import FuncAnimation
from IPython.display import HTML


year = 2000
run_file_name = r"/rdata/ian/pico/paperRuns/finalRuns/global_run_table.pkl"
global_pf_file_name = r"/rdata/ian/pico/paperRuns/global_pf.pkl"

full_run_tab = pd.read_pickle(run_file_name)
global_pf = pd.read_pickle(global_pf_file_name)

global_pf = global_pf.sort_values(by="yield")

igd_ind = GD(global_pf.loc[:,("irr_total", "yield")].values)




In [ ]:

run = 0


selection_masks = [
    np.all([full_run_tab['algorithm'] == "pinsga2", full_run_tab['run'] == run, full_run_tab['DM_range'] == "20to30", full_run_tab['pop_size'] == 30], axis=0),
    np.all([full_run_tab['algorithm'] == "nsga2", full_run_tab['run'] == run, full_run_tab['year'] == 2000], axis=0)]

max_gens = [200, 200]

#labels = ["PI-NSGA-II", "NSGA-II"]
labels = ["NSGA-II"]

run_tabs = []
facecolors = ['none', 'green', 'purple', 'orange']
edgecolors = ['black', 'green', 'purple', 'orange']
markers = ['s', 'o', 'o', 'o']


In [ ]:
for (m, mask) in enumerate(selection_masks):
    run_tabs.append(full_run_tab.loc[mask, ['irr_total', 'yield', 'gen']])



In [ ]:
def filter_dom(run_tab):

    # Perform non-dominated sorting
    nds = NonDominatedSorting()

    minimize_pop = run_tab.copy()
    minimize_pop["yield"] = minimize_pop["yield"] * -1

    fronts = nds.do(minimize_pop.values, only_non_dominated_front=True)

    return run_tab.iloc[fronts,:].copy()
    

In [ ]:
def animate(gen):

    # TODO remove this hard coded 2
    gens = [g + 1 for g in [gen] * 2]

    gens = [gen if gen <= max_gens[g] else max_gens[g] for (g, gen) in enumerate(gens)]

    rt_single_gen = [
        run_tab[run_tab["gen"] == gens[t]] for (t, run_tab) in enumerate(run_tabs)
    ]

    for (d, run_tab) in enumerate(rt_single_gen):

        # Get the Pareto front
        pf = filter_dom(run_tab)

        # Plot the data (first column is f1, second column is f2)
        x = pf.iloc[:,0]
        y = pf.iloc[:,1]

        figures[d].set_offsets(np.c_[x,y])

In [ ]:

fig, ax = plt.subplots(figsize=(8,6))

# Set up empty scatter plots for each of the runs 
figures = []
for (d, run_tab) in enumerate(run_tabs):

    figures.append(ax.scatter([], [],
                label=labels[d],
                facecolors=facecolors[d],
                edgecolors=edgecolors[d],
                marker=markers[d])) 

line = ax.plot(global_pf["irr_total"], global_pf["yield"],
            label="Near-true optima",
            color="red")

# Set up labels and such 
ax.set_xlabel("Irrigation (mm)")
ax.set_ylabel("Yield (kg/ha)")

ax.set_xlim(min(global_pf["irr_total"]),max(global_pf["irr_total"]))
ax.set_ylim(min(global_pf["yield"]),max(global_pf["yield"]))

ax.legend()

ani = FuncAnimation(fig, animate, 
                frames=200, interval=100, repeat=True) 

# Display the animation in the Jupyter Notebook
HTML(ani.to_html5_video())

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

for (d, run_tab) in enumerate(run_tabs):

    max_gen = max(run_tab["gen"]) 

    gd_vals = []

    for gen in range(1,max_gen):

        # Get the Pareto front
        pf = filter_dom(run_tab[run_tab["gen"] == gen])

        gd_vals.append(igd_ind(pf.loc[:,("irr_total", "yield")].values.astype(int)))
    
    ax.plot(gd_vals, label=labels[d], color=edgecolors[d])

    ax.legend()



In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

for (d, run_tab) in enumerate(run_tabs):

    max_gen = max(run_tab["gen"]) 

    gd_vals = []

    for gen in range(1,max_gen):

        # Get the Pareto front
        pf = filter_dom(run_tab[run_tab["gen"] == gen])

        if labels[d] == "PI-NSGA-II":
            pf = pf[np.logical_and(pf["irr_total"] >= 20, pf["irr_total"] <= 30)]

        gd_vals.append(igd_ind(pf.loc[:,("irr_total", "yield")].values.astype(int)))
    
    ax.plot(gd_vals, label=labels[d], color=edgecolors[d])

    ax.legend()
